In [2]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from multiprocessing import Pool
from concurrent.futures import ProcessPoolExecutor
import concurrent.futures
import pandas as pd
from tqdm import tqdm
from scipy.optimize import minimize, basinhopping

plt.rcParams['text.latex.preamble'] = r"\usepackage{amsmath}"
plt.rc('font',**{'family':'sans-serif','sans-serif':['Helvetica']})
plt.rc('text',usetex=True)

import julia
from julia import Main
from julia import Distributed
Main.eval("using QuadGK")

In [3]:
Main.eval('using Distributed')
Main.eval('addprocs(1)') 

array([2], dtype=int64)

In [4]:
Main.eval("""
    @everywhere begin
        using Distributed
        using DelimitedFiles      
        df_LO = readdlm("fixed order/EECLO.csv", ',')
        df_NLO = readdlm("fixed order/EECNLO.csv", ',')
        df_NNLO = readdlm("fixed order/EECNNLO.csv", ',')
    end
""")

Main.include("strong coupling\\constants.jl")
Main.include("strong coupling\\alpha_s.jl")

Main.include("fixed order\\perturbative EEC.jl")
Main.include("fixed order\\SCET.jl")
Main.include("fixed order\\non-singular.jl")
Main.include("fixed order\\analytic.jl")

Main.include("cross section.jl")
#Main.include("NP.jl")
Main.include("parallel/fit.jl")

JuliaError: Exception 'On worker 2:
ArgumentError: Cannot open 'fixed order/EECLO.csv': not a file
Stacktrace:
  [1] #readdlm_auto#9
    @ C:\Users\15652\.julia\packages\DelimitedFiles\aGcsu\src\DelimitedFiles.jl:234
  [2] readdlm_auto
    @ C:\Users\15652\.julia\packages\DelimitedFiles\aGcsu\src\DelimitedFiles.jl:233 [inlined]
  [3] readdlm
    @ C:\Users\15652\.julia\packages\DelimitedFiles\aGcsu\src\DelimitedFiles.jl:170 [inlined]
  [4] readdlm
    @ C:\Users\15652\.julia\packages\DelimitedFiles\aGcsu\src\DelimitedFiles.jl:162
  [5] top-level scope
    @ none:5
  [6] eval
    @ .\boot.jl:385
  [7] #invokelatest#2
    @ .\essentials.jl:892
  [8] invokelatest
    @ .\essentials.jl:889
  [9] #114
    @ C:\Users\15652\AppData\Local\Programs\Julia-1.10.2\share\julia\stdlib\v1.10\Distributed\src\process_messages.jl:303
 [10] run_work_thunk
    @ C:\Users\15652\AppData\Local\Programs\Julia-1.10.2\share\julia\stdlib\v1.10\Distributed\src\process_messages.jl:70
 [11] run_work_thunk
    @ C:\

Range Choice

In [ ]:
χ_lower = 145
χ_upper = 175

In [ ]:
n_replicas = 30
If_fit = True
If_scan = False

Read Data

In [ ]:
#-------------------------------------------------------------------
dataset_names = ["SLD","TOPAZ_595","TOPAZ_533",
                 "TASSO_435","TASSO_348","MARKII","MAC"]
#-------------------------------------------------------------------

dataset_qua = ["OPAL","MAC"]  
 
df_qua = {}

for name in dataset_qua:
    file_path = f"data/{name}.csv"
    df = pd.read_csv(file_path)
    
    χ = df["CHI"]
    EEC = df["EEC"]
    ERROR = df["STAT"]
    
    df1 = pd.DataFrame({"CHI": χ, "EEC": EEC, "ERROR": ERROR})
    df_qua[name] = df1

dataset_ss = ["SLD","DELPHI","TOPAZ_595","TOPAZ_533","TASSO_435","TASSO_348","MARKII"]

df_ss = {}

for name in dataset_ss:
    file_path = f"data/{name}.csv"
    df = pd.read_csv(file_path)
    
    χ = df["CHI"]
    EEC = df["EEC"]
    ERROR = np.sqrt(df["STAT"]**2+df["SYS"]**2)
    
    df1 = pd.DataFrame({"CHI": χ, "EEC": EEC, "ERROR": ERROR})
    df_ss[name] = df1

df = df_qua
df.update(df_ss)

df_truncated = {}
χ = {}
EEC = {}
ERROR = {}

for name in dataset_names:
    df_truncated[name] = df[name][(df[name]["CHI"] <= χ_upper) & (df[name]["CHI"] >= χ_lower)]

    χ[name] = np.array(df_truncated[name]["CHI"])
    EEC[name] = np.array(df_truncated[name]["EEC"])
    ERROR[name] = np.array(df_truncated[name]["ERROR"])

Q_list = {}
Q_list["SLD"]       = 91.2
Q_list["OPAL"]      = 91.2
Q_list["DELPHI"]    = 91.2
Q_list["TOPAZ_595"] = 59.5
Q_list["TOPAZ_533"] = 53.3
Q_list["TASSO_435"] = 43.5
Q_list["TASSO_348"] = 34.8
Q_list["MARKII"]    = 29.0
Q_list["MAC"]       = 29.0

Random data

In [ ]:
EEC_list=[]
df_list=[]
for i in range(n_replicas):
    if If_fit == True:
        df = {}
        for name in dataset_names:
            EEC_gaussian = np.random.normal(loc=EEC[name], scale=ERROR[name])
            df[name] = pd.DataFrame({"CHI": χ[name], "EEC": EEC_gaussian, "ERROR": ERROR[name]})
        df_list.append(df)
    else:
        df = {}
        for name in dataset_names:
            EEC_gaussian = EEC[name]
            df[name] = pd.DataFrame({"CHI": χ[name], "EEC": EEC_gaussian, "ERROR": ERROR[name]})
        df_list.append(df)

In [ ]:
df_list[0]["SLD"]["EEC"]-df_list[1]["SLD"]["EEC"]

0   -0.012111
1    0.006225
2   -0.000618
3    0.017749
4    0.019115
5    0.019006
6    0.015404
7   -0.011727
8   -0.036612
Name: EEC, dtype: float64

Random ratios

In [ ]:
def custom_random():
    n = np.random.uniform(1.0, 2.0)  
    if np.random.rand() < 0.5:
        return n  
    else:
        return 1 / n  

def random_ratios(n):
    ratio_matrix = pd.DataFrame(columns=['μH_ratio', 'μJ_ratio', 'νJ_ratio'])
    
    while len(ratio_matrix) < n:
        
        if If_scan == True:
            μH_ratio, μJ_ratio, νJ_ratio = custom_random(), custom_random(), custom_random()
        else:
            μH_ratio, μJ_ratio, νJ_ratio = 1.0, 1.0, 1.0

        ratio_matrix.loc[len(ratio_matrix)] = [μH_ratio, μJ_ratio, νJ_ratio]
    
    return ratio_matrix

In [ ]:
ratios = random_ratios(n_replicas)
n = len(ratios)

In [ ]:
display(ratios)

,μH_ratio,μJ_ratio,νJ_ratio
0,1.0,1.0,1.0
1,1.0,1.0,1.0
2,1.0,1.0,1.0
3,1.0,1.0,1.0
4,1.0,1.0,1.0
5,1.0,1.0,1.0
6,1.0,1.0,1.0
7,1.0,1.0,1.0
8,1.0,1.0,1.0
9,1.0,1.0,1.0


Chi-Square calculation

In [ ]:
def DELTA_func(ylist,PRED):
    l = len(ylist)
    theory = np.zeros(l)
    for i in range(l):
        simpson = 1/12*(PRED[0][i] + 4*PRED[1][i] + 2*PRED[2][i] + 4*PRED[3][i] + PRED[4][i])
        theory[i] = np.nan_to_num(simpson,nan=0.0)
    return theory - ylist

def CHI2_func(DELTA,ERROR):
    chi_squared = np.sum(np.square(DELTA/ERROR))
    return chi_squared

Define Objective

In [ ]:
def objective(params, df, Q_list, dataset_names, ratio_list):

    #----------
    XLL="N3LL"
    XLO=2 # 1:LO 2:NLO
    #----------
    
    N=5
    chi2 = 0
    length = 0
    
    αs = params[0]
    params_noαs = params[1:]

    for name in dataset_names:

        xlist={}
        xlist_mid=np.array(df[name]["CHI"])
        binsize=abs(xlist_mid[1]-xlist_mid[0]) # Evenly sized bins assumed
        for i in range(N):
            xlist[i]=xlist_mid-binsize/2+i*binsize/4

        ylist=np.array(df[name]["EEC"])
        ERROR_list=np.array(df[name]["ERROR"])

        l = len(xlist_mid)
        length = length + l

        PRED={}

        parameters = params_noαs

        for quartile in range(N):
            PRED[quartile] = Main.model(xlist=xlist[quartile], αs=αs, Q=Q_list[name], XLL=XLL, XLO=XLO, parameters=parameters
                                        , μH_ratio=ratio_list["μH_ratio"], μJ_ratio=ratio_list["μJ_ratio"], νJ_ratio=ratio_list["νJ_ratio"]
                                        )                                                             
        DELTA =  DELTA_func(ylist,PRED)              

        chi_square = CHI2_func(DELTA,ERROR_list)
        chi2 = chi2 + chi_square

    return chi2/length

Define minimization related

In [ ]:
import sys

class ProgressCallback:
    def __init__(self, df, Q_list, dataset_names, ratio_list):
        self.df = df
        self.Q_list = Q_list
        self.dataset_names = dataset_names
        self.ratio_list = ratio_list

    def __call__(self, params):
        chi2_bydof = objective(params, self.df, self.Q_list, self.dataset_names, self.ratio_list)
        sys.stdout.flush()

class MyBounds:
    def __init__(self, xmax, xmin):
        self.xmax = np.array(xmax)
        self.xmin = np.array(xmin)

    def __call__(self, **kwargs):
        x = kwargs["x_new"]
        tmax = bool(np.all(x <= self.xmax))
        tmin = bool(np.all(x >= self.xmin))
        return tmax and tmin

Test

In [ ]:
objective([0.11725829, 0.12684399, 0.44293446, 0.08641648], df_truncated, Q_list, dataset_names, pd.Series({"μH_ratio":1.0,"μJ_ratio":1.0,"νJ_ratio":1.0}))
#[0.1173, 0.127, 0.443, 0.0864] bbstar

1.0123348413316549

Minimization

In [ ]:
results_df = pd.DataFrame(columns=['αs', 'params','chi2/dof'])

initial_params = [0.11725829, 0.12684399, 0.44293446, 0.08641648] #best_params

bounds = [ [0.113 , 0.123], # αs
           #[-1.0 , 1.0  ], # Ω1
            [0.0, 2.0  ], # a1
            [0.0, 2.0  ], # a2
            [0.0, 2.0  ], # a3         
        ]

for i in tqdm(range(n_replicas)): 

    #Ratios
    ith_row = ratios.iloc[i]
    μH_ratio = ith_row['μH_ratio']    
    μJ_ratio = ith_row['μJ_ratio']
    νJ_ratio = ith_row['νJ_ratio']

    ratio_list={}
    ratio_list["μH_ratio"]=μH_ratio
    ratio_list["μJ_ratio"]=μJ_ratio
    ratio_list["νJ_ratio"]=νJ_ratio

    #Fit
    bounds_T = [list(t) for t in zip(*bounds)]

    progress_callback = ProgressCallback(df_list[i], Q_list, dataset_names, ratio_list)

    minimizer_kwargs = {
        "method": "L-BFGS-B",
        "bounds": bounds,
        "args": (df_list[i], Q_list, dataset_names, ratio_list),
        "options": {'maxiter': 100, 'disp': True, "ftol": 10**(-6)},
        "callback": progress_callback
    }

    result = basinhopping(
        objective,
        initial_params,
        minimizer_kwargs=minimizer_kwargs,
        niter=0,
        accept_test=MyBounds(xmax=bounds_T[1],xmin=bounds_T[0])
    )

#-----------------------------------------------------------------------------

    optimal = np.round(result.x,5)
    chi2_per_dof = result.fun

    optimal_params = optimal[1:]

    new_row = pd.Series({
        'αs': optimal[0], 
        'params': optimal_params, 
        'chi2/dof': result.fun,
    })
    results_df = pd.concat([results_df, new_row.to_frame().T], ignore_index=True)
    #print(results_df)

display(results_df)

100%|██████████| 30/30 [23:51<00:00, 47.71s/it]


,αs,params,chi2/dof
0,0.11808,"[0.12579, 0.43967, 0.08237]",2.276294
1,0.11423,"[0.14979, 0.50405, 0.09344]",1.840913
2,0.11772,"[0.11989, 0.41128, 0.1067]",1.93023
3,0.11637,"[0.13363, 0.46131, 0.08958]",1.758465
4,0.11637,"[0.13465, 0.47208, 0.08055]",1.549066
5,0.11784,"[0.02519, 0.42845, 0.10966]",1.864143
6,0.11546,"[0.0, 0.49461, 0.11589]",2.104901
7,0.11525,"[0.15107, 0.50671, 0.08367]",1.867481
8,0.11755,"[0.12728, 0.44365, 0.08729]",2.25728
9,0.11612,"[0.13277, 0.4402, 0.09807]",1.575313


Store Result to a File

In [ ]:
results_df.to_csv("result//bbstar_fit_40.csv", index=False)

In [ ]:
Main.eval('rmprocs(workers())')

<PyCall.jlwrap Task (runnable) @0x000001daf808f840>

KeyboardInterrupt: 